In [ ]:
import requests
import time
import json
import random
import pandas as pd
import numpy as np
import pickle

In [ ]:
DATA_PATH = '../data/all_reviews/all_reviews.csv'
PROCESSED_DATASET_PATH = "../data/all_reviews/processed_dataset.csv"
#MODEL_PATH = '../model/'

In [ ]:
p = 0.20

columns = [
    'recommendationid', 'appid', 'game', 'author_steamid', 'author_num_games_owned', 
    'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 
    'author_playtime_at_review', 'author_last_played', 'voted_up'
]

# Sample 20% of the overall dataset
df = pd.read_csv(PROCESSED_DATASET_PATH, header=0, skiprows= lambda i: i>0 and random.random() > p)

# Only keep games with enough data
game_counts = df.groupby('appid')['author_steamid'].count() # Group by appid, then get how many reviews the game has based on steamid
popular_games = game_counts[game_counts > 5].index # Keep only the games that have more than 200 reviews
df = df[df['appid'].isin(popular_games)] # Remake the original df using only appid's that are in popular_games

# Keep only users who play enough games
user_counts = df.groupby('author_steamid')['appid'].count() # Group by steamid, then get how many games theyve reviewed based on appid
active_users = user_counts[user_counts > 5].index # Keep only the users that have reviewed more than 5 games 
df = df[df['author_steamid'].isin(active_users)] # Remake the df using only steamid's that are in active_users

# Force hours columns to be of type numeric
df['author_playtime_forever'] = pd.to_numeric(df['author_playtime_forever'], errors='coerce')

# Convert hours played into a confidence score or rating
df['author_playtime_forever_normalized'] = np.log(df['author_playtime_forever'] + 1)
df['author_playtime_last_two_weeks_normalized'] = np.log(df['author_playtime_last_two_weeks'] + 1)
df['author_playtime_at_review_normalized'] = np.log(df['author_playtime_at_review'] + 1)


print(f"Final training set size: {len(df)} rows")
print(f"Unique games: {df['appid'].nunique()}")
print(f"Unique users: {df['author_steamid'].nunique()}")
df.head(10)
print(df[df['appid'] == 377160].shape[0])

In [ ]:
user_game_matrix = df.pivot_table(
    index='author_steamid',
    columns='game', 
    values='voted_up'
).fillna(0)

# Convert to a 2D matrix
game_user_matrix = user_game_matrix.values.T

print(game_user_matrix.shape)

In [ ]:
# Rebuild the local dictionary from original dataframe
game_mapping = df[['appid', 'game']].drop_duplicates()
name_to_id_local = {str(row['game']).lower().strip(): row['appid'] for _, row in game_mapping.iterrows()}

In [ ]:
# Declare headers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/91.0.4472.124 Safari/537.36'
}

# Get the exact columns from matrix
matrix_games = list(user_game_matrix.columns)

forbidden_tags = {"Sexual Content", "Hentai", "Nudity", "NSFW", "Anime", "Dating Sim"}

master_tags_dict = {}
master_banned_list = [] 
master_reviews_dict = {} 

print(f"Starting scrape for {len(matrix_games)} games")

for i, game_name in enumerate(matrix_games):
    # Progress tracker
    if i % 100 == 0 and i > 0:
        print(f"   ... Processed {i} / {len(matrix_games)} games. Banned so far: {len(master_banned_list)} ...")

    clean_name = str(game_name).lower().strip()
    app_id = name_to_id_local.get(clean_name)

    # If no ID is found locally, blank everything out safely
    if not app_id:
        master_tags_dict[game_name] = ""
        master_reviews_dict[game_name] = {'positive': 0, 'negative': 0}
        continue

    spy_url = f"https://steamspy.com/api.php?request=appdetails&appid={app_id}"

    try:
        res = requests.get(spy_url, headers=headers, timeout=10)
        
        if res.status_code == 200:
            data = res.json()
            
            # Grab reviews
            pos_reviews = data.get('positive', 0)
            neg_reviews = data.get('negative', 0)
            master_reviews_dict[game_name] = {'positive': pos_reviews, 'negative': neg_reviews}
            
            # NFSW tag filter
            if 'tags' in data and isinstance(data['tags'], dict):
                tags_list = list(data['tags'].keys())
                top_4_tags = tags_list[:4]
                
                if any(bad_tag in top_4_tags for bad_tag in forbidden_tags):
                    master_banned_list.append(game_name)
                    master_tags_dict[game_name] = "" 
                else:
                    master_tags_dict[game_name] = " ".join(tags_list) 
            else:
                master_tags_dict[game_name] = "" 
                
        else:
            master_tags_dict[game_name] = ""
            master_reviews_dict[game_name] = {'positive': 0, 'negative': 0}
            
    except Exception:
        master_tags_dict[game_name] = "" 
        master_reviews_dict[game_name] = {'positive': 0, 'negative': 0}

    time.sleep(1.2) 

print(f"\nScrape complete, total NSFW games banned: {len(master_banned_list)}")

# Save to JSONs
with open('final_steam_tags_dict.json', 'w') as f:
    json.dump(master_tags_dict, f, indent=4)

with open('final_banned_list.json', 'w') as f:
    json.dump(master_banned_list, f)

with open('final_review_scores.json', 'w') as f:
    json.dump(master_reviews_dict, f, indent=4)

print("Saved successfully.")

In [ ]:
with open('name_to_id_local.pkl', 'wb') as f:
    pickle.dump('name_to_id_local', f)